# Amazon Bedrock TTFT: Converse vs Anthropic Messages API

This notebook compares **time-to-first-token (TTFT)** for Claude Opus 4.7 across two different streaming APIs offered by Amazon Bedrock:

| API | Endpoint class | Protocol | Bedrock endpoint |
|-----|---------------|----------|------------------|
| **[ConverseStream](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_ConverseStream.html)** | `BedrockConverseStream` | AWS event-stream | `bedrock-runtime` |
| **[Anthropic Messages](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-messages.html)** | `AnthropicMessagesStream` | SSE (server-sent events) | `bedrock-mantle` |

Both hit the same Opus 4.7 model in the same AWS Region, but the transport layer differs. We use LLMeter's `Runner` to send identical prompts through each path and compare the resulting TTFT distributions.

A `NoCacheCallback` appends a unique prefix to each request to defeat any prompt-level caching on the Mantle endpoint.

In [ ]:
# %pip install "llmeter[anthropic,plotting]<1"

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Anti-caching callback

Adds a unique prefix like `[req-00001-a3f2b]` to the start of the first message in each request before it's sent. This keeps prompt length consistent while preventing cache hits from skewing TTFT metrics.

Handles both payload formats:
- **Converse**: `messages[].content[].text` (text block in a list)
- **Messages API**: `messages[].content` (plain string)

In [ ]:
import uuid
from llmeter.callbacks import Callback


class NoCacheCallback(Callback):
    """Add a unique prefix to each request to defeat prompt caching."""

    def __init__(self):
        self._counter = 0

    async def before_invoke(self, payload: dict) -> None:
        self._counter += 1
        tag = f"[req-{self._counter:05d}-{uuid.uuid4().hex[:5]}] "

        messages = payload.get("messages", [])
        if not messages:
            return

        first_msg = messages[0]
        content = first_msg.get("content")

        # Messages API format: content is a plain string
        if isinstance(content, str):
            first_msg["content"] = tag + content
            return

        # Converse format: content is a list of blocks
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and "text" in block:
                    block["text"] = tag + block["text"]
                    return

## Configure endpoints

Both endpoints target Opus 4.7 in `us-east-1`:

- **Converse** uses `bedrock-runtime` via boto3 with `global.anthropic.claude-opus-4-7`
- **Messages API** uses `bedrock-mantle` via the `anthropic` SDK with `anthropic.claude-opus-4-7`

In [ ]:
from llmeter.endpoints.bedrock import BedrockConverseStream
from llmeter.endpoints.anthropic_messages import AnthropicMessagesStream

REGION = "us-east-1"

# --- Converse API (bedrock-runtime) ---
converse_endpoint = BedrockConverseStream(
    model_id="global.anthropic.claude-opus-4-7",
    region=REGION,
    endpoint_name="converse",
)

# --- Anthropic Messages API (bedrock-mantle) ---
messages_endpoint = AnthropicMessagesStream(
    model_id="anthropic.claude-opus-4-7",
    provider="bedrock-mantle",
    aws_region=REGION,
    endpoint_name="messages-api",
)

## Build payloads

Each API has its own payload format. We use the `create_payload` helpers to
build equivalent requests with the same prompt and `max_tokens`.

In [ ]:
from llmeter.endpoints.bedrock import BedrockBase
from llmeter.endpoints.anthropic_messages import AnthropicMessagesEndpoint

PROMPT = (
    "Explain the key differences between REST and GraphQL APIs. "
    "Cover query flexibility, over-fetching, versioning, and caching."
)
MAX_TOKENS = 512

converse_payload = BedrockBase.create_payload(PROMPT, max_tokens=MAX_TOKENS)
messages_payload = AnthropicMessagesEndpoint.create_payload(PROMPT, max_tokens=MAX_TOKENS)

## Verify both endpoints

In [ ]:
for name, ep, payload in [
    ("Converse", converse_endpoint, converse_payload),
    ("Messages API", messages_endpoint, messages_payload),
]:
    r = ep.invoke(payload)
    assert r.error is None, f"{name} error: {r.error}"
    print(
        f"{name:>12}  "
        f"TTFT={r.time_to_first_token:.3f}s  "
        f"TTLT={r.time_to_last_token:.3f}s  "
        f"in={r.num_tokens_input} out={r.num_tokens_output}"
    )

## Run the comparison

Single client, same prompt, same model, same region. The only variable is
the API path and transport protocol.

In [ ]:
from llmeter.runner import Runner

N_REQUESTS = 10
OUTPUT_PATH = "outputs/opus-4.7-ttft-comparison"

# --- Converse ---
converse_runner = Runner(
    endpoint=converse_endpoint,
    output_path=OUTPUT_PATH,
    callbacks=[NoCacheCallback()],
)
converse_result = await converse_runner.run(
    payload=converse_payload,
    n_requests=N_REQUESTS,
    clients=1,
    run_name="converse",
)

print(f"Converse: {converse_result.total_requests} requests")
print(f"  p50 TTFT: {converse_result.stats['time_to_first_token-p50']:.3f}s")
print(f"  p90 TTFT: {converse_result.stats['time_to_first_token-p90']:.3f}s")

In [ ]:
# --- Messages API ---
messages_runner = Runner(
    endpoint=messages_endpoint,
    output_path=OUTPUT_PATH,
    callbacks=[NoCacheCallback()],
)
messages_result = await messages_runner.run(
    payload=messages_payload,
    n_requests=N_REQUESTS,
    clients=1,
    run_name="messages-api",
)

print(f"Messages API: {messages_result.total_requests} requests")
print(f"  p50 TTFT: {messages_result.stats['time_to_first_token-p50']:.3f}s")
print(f"  p90 TTFT: {messages_result.stats['time_to_first_token-p90']:.3f}s")

## Compare results

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    {
        "Converse": {
            "p50 TTFT (s)": converse_result.stats["time_to_first_token-p50"],
            "p90 TTFT (s)": converse_result.stats["time_to_first_token-p90"],
            "p50 TTLT (s)": converse_result.stats["time_to_last_token-p50"],
            "p90 TTLT (s)": converse_result.stats["time_to_last_token-p90"],
            "Avg output tokens": converse_result.stats.get("num_tokens_output-mean", "N/A"),
            "Requests": converse_result.total_requests,
            "Errors": converse_result.stats.get("errors", 0),
        },
        "Messages API": {
            "p50 TTFT (s)": messages_result.stats["time_to_first_token-p50"],
            "p90 TTFT (s)": messages_result.stats["time_to_first_token-p90"],
            "p50 TTLT (s)": messages_result.stats["time_to_last_token-p50"],
            "p90 TTLT (s)": messages_result.stats["time_to_last_token-p90"],
            "Avg output tokens": messages_result.stats.get("num_tokens_output-mean", "N/A"),
            "Requests": messages_result.total_requests,
            "Errors": messages_result.stats.get("errors", 0),
        },
    }
)

summary

## Plot TTFT distributions

Using LLMeter's built-in `histogram_by_dimension` and `boxplot_by_dimension`
plotting primitives.

In [ ]:
import plotly.graph_objects as go
from llmeter.plotting import histogram_by_dimension, boxplot_by_dimension

# --- TTFT Histogram ---
fig_hist = go.Figure()
fig_hist.add_trace(
    histogram_by_dimension(
        converse_result,
        "time_to_first_token",
        name="Converse API",
        opacity=0.7,
        marker_color="#2196F3",
    )
)
fig_hist.add_trace(
    histogram_by_dimension(
        messages_result,
        "time_to_first_token",
        name="Messages API",
        opacity=0.7,
        marker_color="#FF9800",
    )
)
fig_hist.update_layout(
    title="TTFT Distribution: Converse vs Messages API",
    xaxis_title="Time to First Token (seconds)",
    yaxis_title="Probability",
    barmode="overlay",
    height=400,
    template="plotly_white",
)
fig_hist.show()

In [ ]:
# --- TTFT Box plot ---
fig_box = go.Figure()
fig_box.add_trace(
    boxplot_by_dimension(
        converse_result,
        "time_to_first_token",
        name="Converse API",
        marker=dict(color="#2196F3"),
        boxpoints="all",
        jitter=0.3,
    )
)
fig_box.add_trace(
    boxplot_by_dimension(
        messages_result,
        "time_to_first_token",
        name="Messages API",
        marker=dict(color="#FF9800"),
        boxpoints="all",
        jitter=0.3,
    )
)
fig_box.update_layout(
    title="TTFT: Converse vs Messages API (Bedrock, us-east-1)",
    xaxis_title="Time to First Token (seconds)",
    showlegend=False,
    height=400,
    template="plotly_white",
)
fig_box.show()

In [ ]:
# --- TTLT Histogram ---
fig_ttlt = go.Figure()
fig_ttlt.add_trace(
    histogram_by_dimension(
        converse_result,
        "time_to_last_token",
        name="Converse API",
        opacity=0.7,
        marker_color="#2196F3",
    )
)
fig_ttlt.add_trace(
    histogram_by_dimension(
        messages_result,
        "time_to_last_token",
        name="Messages API",
        opacity=0.7,
        marker_color="#FF9800",
    )
)
fig_ttlt.update_layout(
    title="TTLT Distribution: Converse vs Messages API",
    xaxis_title="Time to Last Token (seconds)",
    yaxis_title="Probability",
    barmode="overlay",
    height=400,
    template="plotly_white",
)
fig_ttlt.show()

## Observations

Same model (Opus 4.7), same region (`us-east-1`), same prompt — the only
variable is the API path:

- **Converse** uses AWS event-stream encoding over HTTP/1.1 via boto3
  (`bedrock-runtime`)
- **Messages API** uses standard SSE over HTTP/2 via the `anthropic` SDK
  (`bedrock-mantle`)

Differences in TTFT may come from:
- Connection setup and TLS negotiation overhead
- Request routing through different Bedrock infrastructure paths
- SDK-level overhead (boto3 SigV4 signing vs anthropic SDK SigV4 signing)
- HTTP/1.1 event-stream vs HTTP/2 SSE framing

To get more statistically significant results, increase `N_REQUESTS` above
or use `run_duration` for time-bound runs.